# Pipeline ETL y Detección de Anomalías — SAP AI Security
**Objetivo:** Prueba de concepto (PoC) para el procesamiento por batches de logs de auditoría y detección de anomalías en tiempo casi real mediante `Isolation Forest`.  


In [6]:

# 1. Configuración del Entorno y Dependencias Backend
import os
import json
from typing import Tuple, List, Dict, Any
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

# Configuración de visualización de tablas en consola / notebook
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

# Ruta base del dataset de logs
DATA_PATH = 'logs/logs.csv'
print('Dependencias cargadas correctamente.')


Dependencias cargadas correctamente.


## 2. Módulo ETL: Ingesta y Agregación Temporal

Este módulo define la función `procesar_batch_etl`, la cual:
1. Valida y transforma la columna `@timestamp` a formato `datetime64[ns, UTC]`.
2. Indexa y ordena temporalmente las observaciones para garantizar consistencia cronológica.
3. Aplica un `resample` en ventanas regulares de **5 minutos** (`5min`).
4. Agrega los KPIs críticos de seguridad y rendimiento de SAP AI:
   - `volumen_total_peticiones`: Carga total de trabajo en el intervalo.
   - `latencia_promedio` y `latencia_maxima`: Tiempos de respuesta (ms) del servicio LLM.
   - `suma_tokens`: Consumo acumulado de tokens de entrada y salida.
   - `conteo_ips_unicas`: Cardinalidad de clientes concurrentes.
5. Realiza imputación determinista de valores nulos para ventanas sin llamadas a modelos.

In [7]:
# 2.1 Definición de la Función ETL
def procesar_batch_etl(df: pd.DataFrame, freq: str = "5min") -> pd.DataFrame:
    """
    Procesa un batch crudo de logs y agrega métricas clave en ventanas temporales.
    
    Parámetros:
    -----------
    df : pd.DataFrame
        Batch de registros con metadatos de peticiones y modelos LLM.
    freq : str, opcional
        Frecuencia de muestreo temporal para resample (por defecto "5min").
        
    Retorna:
    --------
    pd.DataFrame
        DataFrame con índice temporal y columnas agregadas de KPIs.
    """
    if df.empty:
        raise ValueError("El DataFrame de entrada está vacío.")
        
    df_proc = df.copy()
    
    # 1. Normalización y tipado de la marca temporal
    time_col = "@timestamp"
    if time_col not in df_proc.columns:
        raise KeyError(f"Columna obligatoria '{time_col}' no encontrada en el batch.")
        
    df_proc[time_col] = pd.to_datetime(df_proc[time_col], errors="coerce", utc=True)
    df_proc = df_proc.dropna(subset=[time_col])
    df_proc = df_proc.set_index(time_col).sort_index()
    
    # 2. Identificación de columna de latencia (fallback a sap_llm_response_time)
    lat_col = "llm_response_time_ms" if "llm_response_time_ms" in df_proc.columns else "sap_llm_response_time"
    token_col = "llm_total_tokens" if "llm_total_tokens" in df_proc.columns else None
    ip_col = "client_ip" if "client_ip" in df_proc.columns else "_id"
    
    # Asegurar tipos numéricos para cálculos de agregación
    if lat_col in df_proc.columns:
        df_proc[lat_col] = pd.to_numeric(df_proc[lat_col], errors="coerce")
    if token_col and token_col in df_proc.columns:
        df_proc[token_col] = pd.to_numeric(df_proc[token_col], errors="coerce")
        
    # 3. Agregación temporal por resample
    agg_dict = {
        "volumen_total_peticiones": (ip_col, "count"),
        "latencia_promedio": (lat_col, "mean") if lat_col in df_proc.columns else (ip_col, lambda _: 0.0),
        "latencia_maxima": (lat_col, "max") if lat_col in df_proc.columns else (ip_col, lambda _: 0.0),
        "suma_tokens": (token_col, "sum") if token_col else (ip_col, lambda _: 0.0),
        "conteo_ips_unicas": (ip_col, "nunique")
    }
    
    df_ventanas = df_proc.resample(freq).agg(**agg_dict)
    
    # 4. Imputación determinista de nulos en ventanas sin tráfico o sin llamadas LLM
    df_ventanas["latencia_promedio"] = df_ventanas["latencia_promedio"].fillna(0.0).round(2)
    df_ventanas["latencia_maxima"] = df_ventanas["latencia_maxima"].fillna(0.0).round(2)
    df_ventanas["suma_tokens"] = df_ventanas["suma_tokens"].fillna(0.0).astype(float)
    df_ventanas["volumen_total_peticiones"] = df_ventanas["volumen_total_peticiones"].fillna(0).astype(int)
    df_ventanas["conteo_ips_unicas"] = df_ventanas["conteo_ips_unicas"].fillna(0).astype(int)
    
    # Conservar únicamente ventanas con actividad efectiva
    df_ventanas = df_ventanas[df_ventanas["volumen_total_peticiones"] > 0].copy()
    
    return df_ventanas

## 3. Módulo ML: Detección de Anomalías con Isolation Forest

Este módulo encapsula la función `detectar_anomalias`:
- **Algoritmo:** `IsolationForest` de Scikit-learn (método no supervisado basado en particionado recursivo en árboles aleatorios).
- **Configuración:**
  - `contamination=0.02`: Define el 2% superior de instancias con mayor aislamiento como anomalías potenciales.
- **Salidas:**
  - `anomaly_label`: `1` para comportamiento estándar y `-1` para anomalías.
  - `anomaly_score`: Puntuación de decisión continua (`decision_function`), útil para priorizar la severidad de las alertas.

In [8]:
# 3.1 Definición de la Función de Modelado Predictivo

def detectar_anomalias(
    df_ventanas: pd.DataFrame,
    contamination: float = 0.02,
    random_state: int = 42
) -> Tuple[pd.DataFrame, IsolationForest]:
    """
    Entrena un modelo IsolationForest sobre ventanas temporales para predecir anomalías.
    
    Parámetros:
    -----------
    df_ventanas : pd.DataFrame
        DataFrame indexado temporalmente con las métricas agregadas del ETL.
    contamination : float, opcional
        Proporción esperada de valores atípicos en el conjunto de datos (0.02 = 2%).
    random_state : int, opcional
        Semilla aleatoria para reproducibilidad de los árboles.
        
    Retorna:
    --------
    Tuple[pd.DataFrame, IsolationForest]
        - DataFrame enriquecido con columnas 'anomaly_label' (-1 o 1) y 'anomaly_score'.
        - Objeto IsolationForest entrenado (para posterior exportación o inferencia online).
    """
    if df_ventanas.empty:
        raise ValueError("El DataFrame de ventanas está vacío; no es posible entrenar el modelo.")
        
    features: List[str] = [
        "volumen_total_peticiones",
        "latencia_promedio",
        "latencia_maxima",
        "suma_tokens",
        "conteo_ips_unicas"
    ]
    
    # Validar que todas las características requeridas existan
    faltantes = [f for f in features if f not in df_ventanas.columns]
    if faltantes:
        raise KeyError(f"Faltan las siguientes columnas requeridas para el modelo: {faltantes}")
        
    X = df_ventanas[features].copy()
    
    # Inicialización y entrenamiento del modelo Isolation Forest
    modelo = IsolationForest(
        contamination=contamination,
        random_state=random_state,
        n_estimators=100,
        n_jobs=-1
    )
    
    df_resultado = df_ventanas.copy()
    df_resultado["anomaly_label"] = modelo.fit_predict(X)
    df_resultado["anomaly_score"] = modelo.decision_function(X).round(4)
    
    return df_resultado, modelo

## 4. Simulación de la Orquestación (Batch Pipeline)

Este bloque simula el flujo completo de ejecución que orquestará el microservicio en Cloud Foundry:
1. **Ingesta:** Carga de un batch de prueba desde `logs.csv`.
2. **Transformación:** Ejecución de `procesar_batch_etl`.
3. **Inferencia:** Clasificación con `detectar_anomalias`.
4. **Filtrado y Formateo:** Extracción de las ventanas catalogadas con `-1` y generación del payload estructurado para el **agente generador de reportes**.

In [9]:

# 4.1 Ejecución del Pipeline: Procesamiento Completo o por Batch
if __name__ == '__main__' or True:
    print('INICIANDO SIMULACIÓN DEL PIPELINE ETL Y DETECCIÓN DE ANOMALÍAS')
    print('=' * 80)
    
    # Configuración de carga:
    # True  -> Procesa los 1,931,412 registros de toda la semana (~270 MB en RAM)
    # False -> Carga rápida de un batch inicial para pruebas ágiles
    PROCESAR_TODO = True
    BATCH_SIZE = 50000
    
    # Columnas requeridas por el pipeline para optimización de memoria
    COLS_ETL = [
        '@timestamp',
        'client_ip',
        'llm_response_time_ms',
        'sap_llm_response_time',
        'llm_total_tokens'
    ]
    
    if PROCESAR_TODO:
        print(f'Cargando dataset COMPLETO (1.93M registros) desde {DATA_PATH}...')
        df_batch = pd.read_csv(DATA_PATH, usecols=COLS_ETL)
    else:
        print(f'Cargando lote de prueba ({BATCH_SIZE:,} registros) desde {DATA_PATH}...')
        df_batch = pd.read_csv(DATA_PATH, nrows=BATCH_SIZE, usecols=COLS_ETL)
        
    print(f'Datos en memoria: {df_batch.shape[0]:,} filas cargadas exitosamente.')
    
    # 1. Ejecutar módulo ETL
    print("\n[ETL] Ejecutando procesar_batch_etl(freq='5min')...")
    df_ventanas = procesar_batch_etl(df_batch, freq='5min')
    print(f'[ETL] Ventanas temporales generadas: {len(df_ventanas):,} ventanas de 5 min.')
    print(f'[ETL] Cobertura temporal: {df_ventanas.index.min()} a {df_ventanas.index.max()}')
    print('\nPrimeras 3 ventanas ordenadas:')
    print(df_ventanas.head(3))
    
    # 2. Ejecutar módulo de Machine Learning (Isolation Forest)
    print('\n[ML] Ejecutando detectar_anomalias(contamination=0.02, random_state=42)...')
    df_scored, modelo_if = detectar_anomalias(df_ventanas, contamination=0.02, random_state=42)
    
    # 3. Filtrado de ventanas anómalas (etiquetadas con -1)
    ventanas_anomalas = df_scored[df_scored['anomaly_label'] == -1].copy()
    total_ventanas = len(df_scored)
    num_anomalias = len(ventanas_anomalas)
    
    print('=' * 80)
    print('RESUMEN DE EJECUCIÓN DEL PIPELINE')
    print('=' * 80)
    print(f'Total de registros procesados:       {df_batch.shape[0]:,}')
    print(f'Total de ventanas evaluadas:        {total_ventanas:,}')
    print(f'Ventanas catalogadas como normales: {total_ventanas - num_anomalias:,}')
    print(f'Ventanas anómalas detectadas (-1):  {num_anomalias:,} ({num_anomalias / total_ventanas * 100:.2f}%)')
    print('=' * 80)
    
    print('\nDETALLE DE VENTANAS ANÓMALAS FILTRADAS (ORDENADAS POR SEVERIDAD):')
    display_cols = [
        'anomaly_score',
        'volumen_total_peticiones',
        'latencia_promedio',
        'latencia_maxima',
        'suma_tokens',
        'conteo_ips_unicas'
    ]
    ventanas_priorizadas = ventanas_anomalas[display_cols].sort_values('anomaly_score', ascending=True)
    print(ventanas_priorizadas.head(15))


INICIANDO SIMULACIÓN DEL PIPELINE ETL Y DETECCIÓN DE ANOMALÍAS
Cargando dataset COMPLETO (1.93M registros) desde logs/logs.csv...
Datos en memoria: 1,931,412 filas cargadas exitosamente.

[ETL] Ejecutando procesar_batch_etl(freq='5min')...
[ETL] Ventanas temporales generadas: 2,017 ventanas de 5 min.
[ETL] Cobertura temporal: 2026-04-11 00:00:00+00:00 a 2026-04-18 00:00:00+00:00

Primeras 3 ventanas ordenadas:
                           volumen_total_peticiones  latencia_promedio  latencia_maxima  suma_tokens  conteo_ips_unicas
@timestamp                                                                                                             
2026-04-11 00:00:00+00:00                       594            8720.06         34713.95     711424.0                105
2026-04-11 00:05:00+00:00                       546            7923.14         34885.24     721194.0                104
2026-04-11 00:10:00+00:00                       612            8867.14         34867.94     682179.0      

## 5. Formato de Salida para el Agente Generador de Reportes

Para desacoplar este microservicio y permitir que el agente generador de reportes (LLM Agent) consuma las anomalías, estructuramos un generador de payloads JSON listo para envío vía REST API o Message Queue (RabbitMQ / Kafka).

In [10]:
# 5.1 Generación de Payload JSON para el Agente Generador de Reportes

def generar_payload_alertas(df_anomalias: pd.DataFrame) -> List[Dict[str, Any]]:
    """
    Convierte las ventanas anómalas en una lista de diccionarios JSON listos para el agente.
    """
    alertas = []
    for timestamp, row in df_anomalias.iterrows():
        alerta = {
            "timestamp_inicio": timestamp.isoformat(),
            "duracion_ventana": "5min",
            "anomaly_score": float(row["anomaly_score"]),
            "metricas": {
                "volumen_peticiones": int(row["volumen_total_peticiones"]),
                "latencia_promedio_ms": float(row["latencia_promedio"]),
                "latencia_maxima_ms": float(row["latencia_maxima"]),
                "total_tokens_consumidos": float(row["suma_tokens"]),
                "ips_unicas_activas": int(row["conteo_ips_unicas"])
            },
            "severidad": "CRITICAL" if row["anomaly_score"] < -0.1 else "WARNING"
        }
        alertas.append(alerta)
    return alertas

# Ejemplo del payload generado para el primer incidente detectado
payload_demo = generar_payload_alertas(ventanas_priorizadas.head(3))
print("Ejemplo de Payload JSON para el Agente de Reportes (primeras alertas):")
print(json.dumps(payload_demo, indent=2))

Ejemplo de Payload JSON para el Agente de Reportes (primeras alertas):
[
  {
    "timestamp_inicio": "2026-04-18T00:00:00+00:00",
    "duracion_ventana": "5min",
    "anomaly_score": -0.2417,
    "metricas": {
      "volumen_peticiones": 3,
      "latencia_promedio_ms": 2358.69,
      "latencia_maxima_ms": 3797.08,
      "total_tokens_consumidos": 4508.0,
      "ips_unicas_activas": 3
    },
    "severidad": "CRITICAL"
  },
  {
    "timestamp_inicio": "2026-04-16T12:40:00+00:00",
    "duracion_ventana": "5min",
    "anomaly_score": -0.0885,
    "metricas": {
      "volumen_peticiones": 567,
      "latencia_promedio_ms": 7512.72,
      "latencia_maxima_ms": 33604.85,
      "total_tokens_consumidos": 726166.0,
      "ips_unicas_activas": 105
    },
    "severidad": "WARNING"
  },
  {
    "timestamp_inicio": "2026-04-15T20:10:00+00:00",
    "duracion_ventana": "5min",
    "anomaly_score": -0.0746,
    "metricas": {
      "volumen_peticiones": 514,
      "latencia_promedio_ms": 8231.14,
  